# 2-Layer Neural Network From Scratch (NumPy only)

**Goal:** derive the forward and backward pass of a 2-layer network by hand, implement and train it with NumPy (no autograd), then check it against a reference implementation.

**Plan for this notebook**
1. Paper derivation: forward pass, loss, backward pass, with the shape of every matrix. *(this part)*
2. Forward pass and loss in NumPy. *(this part)*
3. Backward pass, parameter updates and training loop. *(next)*

scikit-learn is used only to generate the toy dataset. The network itself uses NumPy only.

## 1. Setup and notation

- $m$ = number of samples, $d$ = number of input features, $h$ = number of hidden neurons. Here $m = 200$, $d = 2$, $h = 4$.
- Each **row** is one sample. Biases are added to every row (broadcasting).

| Symbol | Meaning | Shape |
|---|---|---|
| $X$ | inputs | $(m, d)$ |
| $y$ | targets (0 or 1) | $(m, 1)$ |
| $W_1$ | first-layer weights | $(d, h)$ |
| $b_1$ | first-layer bias | $(1, h)$ |
| $W_2$ | second-layer weights | $(h, 1)$ |
| $b_2$ | second-layer bias | $(1, 1)$ |
| $Z_1$ | hidden pre-activation | $(m, h)$ |
| $A_1$ | hidden activation | $(m, h)$ |
| $Z_2$ | output pre-activation (logit) | $(m, 1)$ |
| $\hat{y}$ | predicted probability | $(m, 1)$ |

I write $dV$ for $\dfrac{\partial L}{\partial V}$. It always has the **same shape as $V$**, which is a useful check on every formula below.

## 2. Forward pass and loss

$$Z_1 = X W_1 + b_1 \qquad A_1 = \mathrm{ReLU}(Z_1) \qquad Z_2 = A_1 W_2 + b_2 \qquad \hat{y} = \sigma(Z_2)$$

Shapes: $(m,d)(d,h) \to (m,h)$ for $Z_1$, and $(m,h)(h,1) \to (m,1)$ for $Z_2$.

Flow of the computation:

`X → Z1 → A1 → Z2 → ŷ → L`

- $\mathrm{ReLU}(z) = \max(0, z)$ (element-wise)
- $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ (element-wise), squashes the logit into a probability in $(0,1)$

**Loss: binary cross-entropy (mean over the $m$ samples)**

$$L = -\frac{1}{m}\sum_{i=1}^{m}\Big[\,y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\,\Big]$$

If $y_i = 1$ the term is $-\log \hat{y}_i$ (small when $\hat{y}_i$ is close to 1). If $y_i = 0$ it is $-\log(1-\hat{y}_i)$ (small when $\hat{y}_i$ is close to 0). Confident wrong answers are punished heavily.

## 3. Backward pass (chain rule, from the loss back to each parameter)

The backward pass follows the forward arrows in reverse:

`L ← ŷ ← Z2 ← (A1, W2, b2)` and `A1 ← Z1 ← (X, W1, b1)`

At every arrow I multiply the gradient arriving from the right by the **local derivative** of that step.

### 3.1 Derivative of the sigmoid

With $\sigma(z) = (1+e^{-z})^{-1}$:

$$\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \underbrace{\frac{1}{1+e^{-z}}}_{\sigma(z)}\cdot\underbrace{\frac{e^{-z}}{1+e^{-z}}}_{1-\sigma(z)} = \sigma(z)\,\big(1-\sigma(z)\big)$$

So $\dfrac{\partial \hat{y}_i}{\partial Z_{2,i}} = \hat{y}_i(1-\hat{y}_i)$.

### 3.2 Loss with respect to the prediction ($L \to \hat{y}$)

$$\frac{\partial L}{\partial \hat{y}_i} = -\frac{1}{m}\left[\frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i}\right] = -\frac{1}{m}\cdot\frac{y_i(1-\hat{y}_i) - (1-y_i)\hat{y}_i}{\hat{y}_i(1-\hat{y}_i)} = \frac{1}{m}\cdot\frac{\hat{y}_i - y_i}{\hat{y}_i(1-\hat{y}_i)}$$

### 3.3 Prediction to logit ($\hat{y} \to Z_2$): the key simplification

$$dZ_{2,i} = \frac{\partial L}{\partial \hat{y}_i}\cdot\frac{\partial \hat{y}_i}{\partial Z_{2,i}} = \frac{1}{m}\cdot\frac{\hat{y}_i - y_i}{\hat{y}_i(1-\hat{y}_i)}\cdot\hat{y}_i(1-\hat{y}_i) = \frac{1}{m}(\hat{y}_i - y_i)$$

The $\hat{y}(1-\hat{y})$ terms cancel, so in matrix form:

$$\boxed{dZ_2 = \frac{1}{m}(\hat{y} - y)} \qquad \text{shape } (m,1)$$

The gradient at the output is just the **prediction error** (divided by $m$).

### 3.4 Logit to layer-2 parameters ($Z_2 \to W_2, b_2$)

For sample $i$: $Z_{2,i} = \sum_{k=1}^{h} A_{1,ik}\,W_{2,k} + b_2$. So $\dfrac{\partial Z_{2,i}}{\partial W_{2,k}} = A_{1,ik}$ and $\dfrac{\partial Z_{2,i}}{\partial b_2} = 1$.

$W_2$ and $b_2$ are shared by all samples, so their gradients **sum** the contributions of every sample:

$$dW_{2,k} = \sum_{i=1}^{m} dZ_{2,i}\,A_{1,ik} \;\;\Rightarrow\;\; \boxed{dW_2 = A_1^{\top}\, dZ_2} \qquad (h,m)(m,1) \to (h,1)$$

$$\boxed{db_2 = \sum_{i=1}^{m} dZ_{2,i}} \qquad \text{shape } (1,1)$$

### 3.5 Logit to hidden activation ($Z_2 \to A_1$)

$\dfrac{\partial Z_{2,i}}{\partial A_{1,ik}} = W_{2,k}$, so $dA_{1,ik} = dZ_{2,i}\,W_{2,k}$:

$$\boxed{dA_1 = dZ_2\, W_2^{\top}} \qquad (m,1)(1,h) \to (m,h)$$

### 3.6 Hidden activation to hidden pre-activation ($A_1 \to Z_1$)

$A_1 = \mathrm{ReLU}(Z_1)$ element-wise, and $\mathrm{ReLU}'(z) = 1$ if $z > 0$, else $0$. So:

$$\boxed{dZ_1 = dA_1 \odot \mathbf{1}[Z_1 > 0]} \qquad \text{shape } (m,h)$$

($\odot$ is element-wise multiplication.) The error flows back only through neurons that were active for that sample. For $z \le 0$ the ReLU is flat, so changing $Z_1$ there does not change the loss.

### 3.7 Hidden pre-activation to layer-1 parameters ($Z_1 \to W_1, b_1$)

$Z_{1,ik} = \sum_{j=1}^{d} X_{ij}\,W_{1,jk} + b_{1,k}$. Same pattern as 3.4, summing over samples:

$$\boxed{dW_1 = X^{\top}\, dZ_1} \qquad (d,m)(m,h) \to (d,h)$$

$$\boxed{db_1 = \text{column sums of } dZ_1} \qquad \text{shape } (1,h)$$

## 4. Summary: which chain-rule factor is used at each arrow

| Arrow | Chain-rule step | Result | Shape |
|---|---|---|---|
| $\hat{y} \to Z_2$ | $\dfrac{\partial L}{\partial \hat{y}}\cdot\dfrac{\partial \hat{y}}{\partial Z_2}$ (the $\hat{y}(1-\hat{y})$ cancels) | $dZ_2 = \frac{1}{m}(\hat{y}-y)$ | $(m,1)$ |
| $Z_2 \to W_2, b_2$ | $dZ_2 \cdot \dfrac{\partial Z_2}{\partial W_2}$, with $\dfrac{\partial Z_2}{\partial W_2} = A_1$ | $dW_2 = A_1^\top dZ_2$, $\;db_2 = \sum_i dZ_{2,i}$ | $(h,1)$, $(1,1)$ |
| $Z_2 \to A_1$ | $dZ_2 \cdot \dfrac{\partial Z_2}{\partial A_1}$, with $\dfrac{\partial Z_2}{\partial A_1} = W_2$ | $dA_1 = dZ_2 W_2^\top$ | $(m,h)$ |
| $A_1 \to Z_1$ | $dA_1 \cdot \dfrac{\partial A_1}{\partial Z_1}$, with ReLU$' = \mathbf{1}[Z_1>0]$ | $dZ_1 = dA_1 \odot \mathbf{1}[Z_1>0]$ | $(m,h)$ |
| $Z_1 \to W_1, b_1$ | $dZ_1 \cdot \dfrac{\partial Z_1}{\partial W_1}$, with $\dfrac{\partial Z_1}{\partial W_1} = X$ | $dW_1 = X^\top dZ_1$, $\;db_1 = \sum_i dZ_{1,i}$ | $(d,h)$, $(1,h)$ |

**The full chain for $W_1$** (what I should be able to write unprompted):

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial \hat{y}}\cdot\frac{\partial \hat{y}}{\partial Z_2}\cdot\frac{\partial Z_2}{\partial A_1}\cdot\frac{\partial A_1}{\partial Z_1}\cdot\frac{\partial Z_1}{\partial W_1}$$

**Parameter updates** (gradient descent, same rule as the first notebook):

$$W_1 := W_1 - \alpha\, dW_1 \quad b_1 := b_1 - \alpha\, db_1 \quad W_2 := W_2 - \alpha\, dW_2 \quad b_2 := b_2 - \alpha\, db_2$$

**Shape check:** every gradient must have the same shape as its parameter. If a transpose is confusing, check the shapes.